# Project 2

For this project, I wanted to explore the relationship between internet access and economic development. 

I will focus on the following question: **Do countries with higher internet penetration also tend to have higher levels of income?**


Internet connectivity is a key component for economic growth, access to information, and participation in the global economy. GDP per capita is a widely used (though imperfect) measure of economic development.

My goal is to examine whether there is a visible relationship between these two indicators across countries.


Data sources:
I am working with **two datasets**, both from the **World Bank’s World Development Indicators**, a primary data source:

- **Internet Users (% of population)**: https://data.worldbank.org/indicator/IT.NET.USER.ZS

- **GDP per Capita (USD)**: https://data.worldbank.org/indicator/NY.GDP.PCAP.CD

These datasets are available in CSV format.

I will walk through the cleaning steps, merge the datasets, and then create visualizations, including a derived indicator, to better understand the relationship between connectivity and income.

## Loading the World Bank Data

The World Bank CSVs contain many columns, metadata rows, and region aggregates.  
Before merging the datasets, I load each one and extract:

- "Country Name"
- "Country Code" (ISO-3)
- The year **2020**, which appears consistently in both datasets

I also remove non-country aggregates such as “World”, “High income”, “Europe & Central Asia”, etc., since these are not individual countries.


In [50]:
import pandas as pd

# ---------------------------------------------
# Load Internet Users Dataset
# ---------------------------------------------
internet_raw = pd.read_csv("API_IT.NET.USER.ZS_DS2_en_csv_v2_129784.csv", skiprows=4)

# Keep only relevant columns and rename
internet = internet_raw[["Country Name", "Country Code", "2020"]].rename(
    columns={"2020": "Internet Users (%)"}
)

# Drop rows with missing values for Internet users
internet = internet.dropna(subset=["Internet Users (%)"])

# Make sure the column is numeric
internet["Internet Users (%)"] = pd.to_numeric(
    internet["Internet Users (%)"], errors="coerce"
)
internet = internet.dropna(subset=["Internet Users (%)"])

internet.head()


,Country Name,Country Code,Internet Users (%)
1,Africa Eastern and Southern,AFE,25.0000
2,Afghanistan,AFG,17.0485
3,Africa Western and Central,AFW,31.7000
4,Angola,AGO,36.6347
5,Albania,ALB,72.2377


In [51]:
# ---------------------------------------------
# Load GDP per Capita Dataset
# ---------------------------------------------
gdp_raw = pd.read_csv("API_NY.GDP.PCAP.CD_DS2_en_csv_v2_252771.csv", skiprows=4)

# Keep only relevant columns and rename
gdp = gdp_raw[["Country Name", "Country Code", "2020"]].rename(
    columns={"2020": "GDP per Capita (USD)"}
)

# Drop rows with missing GDP
gdp = gdp.dropna(subset=["GDP per Capita (USD)"])

# Make sure GDP is numeric
gdp["GDP per Capita (USD)"] = pd.to_numeric(
    gdp["GDP per Capita (USD)"], errors="coerce"
)
gdp = gdp.dropna(subset=["GDP per Capita (USD)"])

gdp.head()

,Country Name,Country Code,GDP per Capita (USD)
0,Aruba,ABW,22855.932320
1,Africa Eastern and Southern,AFE,1344.103210
2,Afghanistan,AFG,510.787063
3,Africa Western and Central,AFW,1680.039332
4,Angola,AGO,1449.922867


In [66]:
# ---------------------------------------------
# Remove aggregates (World, regions, income groups)
# ---------------------------------------------
aggregates = [
    "World",
    "High income",
    "Low income",
    "Lower middle income",
    "Upper middle income",
    "Low & middle income",
    "Middle income",
    "Euro area",
    "European Union",
    "OECD members",
    "Arab World",
    "Caribbean small states",
    "Central Europe and the Baltics",
    "East Asia & Pacific",
    "East Asia & Pacific (IDA & IBRD countries)",
    "East Asia & Pacific (excluding high income)",
    "Europe & Central Asia",
    "Europe & Central Asia (IDA & IBRD countries)",
    "Europe & Central Asia (excluding high income)",
    "Latin America & Caribbean",
    "Latin America & Caribbean (excluding high income)",
    "Latin America & the Caribbean (IDA & IBRD countries)",
    "Middle East & North Africa",
    "Middle East, North Africa, Afghanistan & Pakistan (excluding high income)",
    "North America",
    "South Asia",
    "South Asia (IDA & IBRD)",
    "Sub-Saharan Africa",
    "Sub-Saharan Africa (IDA & IBRD countries)",
    "Sub-Saharan Africa (excluding high income)",
    "Africa Eastern and Southern",
    "Africa Western and Central",
    "Other small states",
    "Small states",
    "Pacific island small states",
    "Heavily indebted poor countries (HIPC)",
    "Least developed countries: UN classification",
]

internet = internet[~internet["Country Name"].isin(aggregates)]
gdp = gdp[~gdp["Country Name"].isin(aggregates)]

# Keep only rows where "Country Code" is exactly three letters (actual countries)
internet = internet[internet["Country Code"].str.len() == 3]
gdp = gdp[gdp["Country Code"].str.len() == 3]

# Check
print("Internet users shape after cleaning:", internet.shape)
print("GDP per capita shape after cleaning:", gdp.shape)

internet.head(), gdp.head()

Internet users shape after cleaning: (194, 3)
GDP per capita shape after cleaning: (221, 3)


(           Country Name Country Code  Internet Users (%)
 2           Afghanistan          AFG             17.0485
 4                Angola          AGO             36.6347
 5               Albania          ALB             72.2377
 6               Andorra          AND             93.2056
 8  United Arab Emirates          ARE            100.0000,
   Country Name Country Code  GDP per Capita (USD)
 0        Aruba          ABW          22855.932320
 2  Afghanistan          AFG            510.787063
 4       Angola          AGO           1449.922867
 5      Albania          ALB           5370.777500
 6      Andorra          AND          37361.090067)

## Merging the Datasets

To analyze the relationship between the two variables, I merge the cleaned Internet Users and GDP datasets on:

- "Country Name"
- "Country Code"

This gives me a single dataframe ("df") where each row represents a country with:

- Internet Users (%)
- GDP per Capita (USD)

Once merged, I can create visualizations that reflect both datasets simultaneously.

In [67]:
# ---------------------------------------------
# Merge Internet Users and GDP per Capita
# ---------------------------------------------
df = pd.merge(internet, gdp, on=["Country Name", "Country Code"], how="inner")

df.head()

,Country Name,Country Code,Internet Users (%),GDP per Capita (USD)
0,Afghanistan,AFG,17.0485,510.787063
1,Angola,AGO,36.6347,1449.922867
2,Albania,ALB,72.2377,5370.777500
3,Andorra,AND,93.2056,37361.090067
4,United Arab Emirates,ARE,100.0000,37173.875409


## Scatterplot: Internet Use vs GDP per Capita

I start with a simple scatterplot to explore whether countries with higher internet penetration tend to have higher income levels.

Each point represents a country:

- **x-axis:** Internet Users (% of population)  
- **y-axis:** GDP per Capita (USD)

This chart offers a direct look at the relationship between digital connectivity and economic development.

In [68]:
# ---------------------------------------------
# Visualization: Internet Users vs GDP per Capita
# ---------------------------------------------
import plotly.express as px
from IPython.display import HTML

fig = px.scatter(
    df,
    x="Internet Users (%)",
    y="GDP per Capita (USD)",
    hover_name="Country Name",
    title="Internet Use vs GDP per Capita (2020)",
    opacity=0.7,
    labels={
        "Internet Users (%)": "Internet Users (% of population)",
        "GDP per Capita (USD)": "GDP per Capita (USD)",
    },
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

## Log GDP Visualization

Because GDP per capita varies widely across countries, I also show the relationship using log(GDP per capita). This transformation reduces skewness and makes the pattern much easier to see in a scatterplot.

In [69]:
# ---------------------------------------------
# Log-transformed GDP plot
# ---------------------------------------------
import numpy as np

df["log_GDP_per_Capita"] = np.log(df["GDP per Capita (USD)"])

fig_log = px.scatter(
    df,
    x="Internet Users (%)",
    y="log_GDP_per_Capita",
    hover_name="Country Name",
    title="Internet Use vs log(GDP per Capita) (2020)",
    opacity=0.7,
    labels={
        "Internet Users (%)": "Internet Users (% of population)",
        "log_GDP_per_Capita": "Log GDP per Capita (USD)",
    },
)

HTML(fig_log.to_html(include_plotlyjs="cdn", full_html=False))

## Summary Statistics

To support the visual findings, I compute simple summary statistics and the correlation between GDP per capita and Internet Users (%).

In [70]:
# ---------------------------------------------
# Summary statistics and correlation
# ---------------------------------------------

# Basic summary stats
df[["Internet Users (%)", "GDP per Capita (USD)"]].describe()

,Internet Users (%),GDP per Capita (USD)
count,191.000000,191.000000
mean,63.853775,15503.803842
std,26.032507,24853.536216
min,7.400000,210.008140
25%,43.531450,2124.479580
50%,70.483400,5463.645153
75%,85.041300,17725.482660
max,100.000000,176891.886538


### Internet Users (%)
- Countries range from **7%** to **100%** internet penetration.
- The median country has about **70%** of its population online.
- The distribution is fairly wide, with many middle-income countries falling between 40–85%.

### GDP per Capita (USD)
- GDP per capita ranges from about **$210** to **$176,891**, an extremely wide spread.
- The median country has a GDP per capita around **$5,463**, much lower than the mean (approximately $14,773), reflecting strong right-skew due to a few very wealthy countries.
- This skew is exactly why the log(GDP) visualization is useful.

These summary numbers confirm that both variables vary dramatically across countries, which helps explain why the scatterplots show such a clear, upward-sloping relationship.

In [71]:
# Correlation between Internet use and GDP per capita
df[["Internet Users (%)", "GDP per Capita (USD)"]].corr()

,Internet Users (%),GDP per Capita (USD)
Internet Users (%),1.000000,0.562165
GDP per Capita (USD),0.562165,1.000000


### Correlation
The correlation between Internet Users (%) and GDP per Capita is approximately **0.56**, a moderately strong positive relationship.

This means that, in general, countries with higher internet-access rates tend to have higher incomes.  
However, the correlation is far from perfect, suggesting that factors like policy, geography, infrastructure, and inequality also affect digital inclusion.


## Creating a Derived Metric: GDP per 1% of Internet Penetration

To create a more nuanced visualization, I compute a derived metric that combines both datasets:

**GDP_per_InternetPoint = GDP per Capita (USD) / Internet Users (%)**

This tells us **how much GDP corresponds to each percentage point of internet penetration**.

Why this metric?

- Countries with **high GDP but low internet access** get **high values**  
- Countries with **widespread internet access relative to income** get **low values**

This helps identify mismatches between economic development and digital inclusion.

In [72]:
df["GDP_per_InternetPoint"] = df["GDP per Capita (USD)"] / df["Internet Users (%)"]

# Clip color range to avoid a few extreme outliers dominating the scale
low, high = df["GDP_per_InternetPoint"].quantile([0.05, 0.95])

fig = px.choropleth(
    df,
    locations="Country Code",  # ISO-3 codes
    color="GDP_per_InternetPoint",
    hover_name="Country Name",
    hover_data={
        "GDP per Capita (USD)": ":,.0f",
        "Internet Users (%)": ":.1f",
        "GDP_per_InternetPoint": ":.1f",
        "Country Code": False,  # hide code
    },
    range_color=(low, high),  # for better contrast
    title="GDP per 1 Percentage Point of Internet Penetration (2020)",
    labels={"GDP_per_InternetPoint": "USD per 1% Internet"},
    color_continuous_scale="Viridis",  # Plotly color scale
)

fig.update_layout(
    geo=dict(showframe=False, showcoastlines=True, projection_type="natural earth"),
    coloraxis_colorbar=dict(title="USD per 1% Internet"),
)

HTML(fig.to_html(include_plotlyjs="cdn", full_html=False))

## Choropleth Map: GDP per 1% of Internet Penetration

The choropleth map visualizes the derived metric across the world.

How to read it:

- **Bright yellow (high values):** Wealthy countries with relatively low internet penetration.  
  These may have high GDP but uneven digital access.

- **Dark purple (low values):** Countries where internet penetration is high relative to income.  
  These nations have widespread digital access even at lower income levels.

This map highlights digital inequality and shows where connectivity lags behind economic development.


# Key Takeaways

There is a **clear positive relationship** between Internet Users (%) and GDP per Capita: 
- Countries with higher digital penetration generally have higher levels of economic development.

The derived metric (GDP per 1% Internet penetration) reveals important exceptions:  
- Some countries are wealthy but have relatively low internet adoption, while others have high connectivity relative to their income.

This suggests that economic development and digital access do not always progress together. 
- Policies targeting digital inclusion may help countries maximize the benefits of economic growth.

More broadly, this project demonstrates how combining datasets and creating derived metrics can reveal patterns not visible from raw data alone.


## Caveats and Limitations

A few important notes:

- GDP per capita is an imperfect proxy for well-being or economic development.
- Internet Users (%) does not measure speed, affordability, or quality of access.
- This analysis focuses only on **2020**. Patterns may differ across years or be influenced by COVID-19 conditions.
- The derived metric is exploratory, it does not imply causality.

Still, the data provides useful insights into global digital inequality.

If expanded, this analysis could include:

- More years of data  
- Controlling for region or income group  
- Adding population or education variables  